# 11 · Auto Loader → Bronze

**Who this is for:** you, after notebook 10b (Azure SQL → Bronze).
**Runs on:** Databricks serverless (costs credits: keep it short, detach when done). Run the pipeline `heavden-medallion` once first (`databricks bundle run medallion -t dev`), so the Bronze tables exist.

**By the end you will be able to:**
- explain the **second ingestion pattern**: files → landing volume → Auto Loader → Bronze (the first was database → Change Tracking → Bronze)
- say what **schema hints**, **schema evolution** and **`_rescued_data`** each protect against
- trace any Bronze row back to the file it came from
- run Auto Loader yourself in a sandbox and watch it handle a bad value and a new field

The pipeline code is `databricks/src/pipelines/medallion/bronze.py`; the options and hints it uses come from `heavden.pipelines.bronze`, imported here.

## 1. Setup

In [ ]:
import sys

# Find the repo folder this notebook was deployed into, and import the real code from it.
notebook_path = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
repo_root = "/Workspace" + notebook_path.split("/notebooks/databricks/")[0]
sys.path.insert(0, f"{repo_root}/databricks/src")

from heavden.pipelines import bronze

dbutils.widgets.text("catalog", "heavden_dev")
CATALOG = dbutils.widgets.get("catalog")
LANDING = f"/Volumes/{CATALOG}/bronze/landing"
print(repo_root, CATALOG, LANDING)

## 2. What landed

The generator (`generator/landing.py`) writes one newline-delimited JSON file per simulated hour: `vitals/` and `outcomes/`. A device sends `device_id` and never a patient; a missing field is left out of the object, not sent as `null`.

The volume `bronze.landing` is a window onto the ADLS folder `landing/<env>/`: Unity Catalog governs who can read it, and the files never get copied.

In [ ]:
files = dbutils.fs.ls(f"{LANDING}/vitals/")
print(len(files), "vitals files, e.g.", files[0].name)
print(dbutils.fs.head(files[0].path, 400))

## 3. Schema hints: fix what you know, infer the rest

Left alone, Auto Loader infers JSON types from a sample: `heart_rate` would become `BIGINT`, until a file arrives with `88.5` in it. **Schema hints** pin the types we already know. Anything new is still inferred.

In [ ]:
print(bronze.AUTO_LOADER_OPTIONS)
print(bronze.VITALS_HINTS)

Three things protect Bronze from surprises:

| Surprise | What happens | Option |
|---|---|---|
| A value that doesn't fit its type (`"heart_rate": "high"`) | Row kept; the value goes to `_rescued_data` as JSON | rescued data column (on by default with inference) |
| A field nobody announced (`etco2` after a firmware update) | New column added; the stream stops once, and the next run picks it up | `cloudFiles.schemaEvolutionMode = addNewColumns` |
| The same file listed again | Skipped: Auto Loader remembers every file it has read (checkpoint) | built in |

Bronze never throws data away. Deciding what's valid is Silver's job (notebook 12).

## 4. The Bronze tables the pipeline wrote

In [ ]:
display(
    spark.sql(f"""
    SELECT regexp_extract(_source_file, '[^/]+$', 0) AS file, count(*) AS readings,
           min(ts) AS first_ts, max(ts) AS last_ts, max(_ingest_ts) AS ingested
    FROM {CATALOG}.bronze.vitals_raw
    GROUP BY 1 ORDER BY 1 LIMIT 10
""")
)

In [ ]:
display(
    spark.sql(f"""
    SELECT count(*) AS rows, count(_rescued_data) AS rescued, count(etco2) AS with_etco2,
           count(DISTINCT _source_file) AS files
    FROM {CATALOG}.bronze.vitals_raw
""")
)  # rescued = 0 for clean simulated data

## 5. Auto Loader in a sandbox

The cells below run Auto Loader **outside** the pipeline, on a folder the pipeline never reads (`_tutorial/`), into a scratch table. They write two small files: one with a bad value, then one with a new field. Switch `SANDBOX` on to run them; the last cell cleans up.

In [ ]:
SANDBOX = False  # set True to run sections 5-6 (a few seconds of serverless compute)

TUTORIAL = f"{LANDING}/_tutorial"
TABLE = f"{CATALOG}.bronze.tutorial_vitals"


def write(name, lines):
    dbutils.fs.put(f"{TUTORIAL}/vitals/{name}", "\n".join(lines) + "\n", overwrite=True)


def load():
    """One Auto Loader run: read what's new, then stop (availableNow)."""
    reader = (
        spark.readStream.format("cloudFiles")
        .options(**bronze.AUTO_LOADER_OPTIONS)
        .option("cloudFiles.schemaHints", bronze.VITALS_HINTS)
        .option(
            "cloudFiles.schemaLocation", f"{TUTORIAL}/_schema"
        )  # the pipeline manages this itself
        .load(f"{TUTORIAL}/vitals/")
    )
    query = (
        bronze.with_lineage(reader)
        .writeStream.option("checkpointLocation", f"{TUTORIAL}/_checkpoint")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(TABLE)
    )
    query.awaitTermination()


if SANDBOX:
    write(
        "a.json",
        [
            '{"device_id":"DEV-T-1","ts":"2026-11-01T08:00:00Z","heart_rate":88,"spo2":97}',
            '{"device_id":"DEV-T-1","ts":"2026-11-01T08:05:00Z","heart_rate":"high","spo2":96}',
        ],
    )
    load()
    display(spark.table(TABLE).select("device_id", "ts", "heart_rate", "_rescued_data"))

`"high"` doesn't fit `DOUBLE`, so `heart_rate` is `NULL` and `_rescued_data` keeps the original value (with the file it came from). Nothing was lost, and nothing broke.

## 6. A new field: schema evolution

Now a device sends a field nobody announced, `co2_ppm`. (`etco2`, the real firmware change, is already in the hints, so it never surprises Bronze.) With `addNewColumns`, the first run **stops with an error** after recording the new schema. That's deliberate: a schema change is something to notice. The next run continues with the new column. (The pipeline restarts automatically, so there you only see a retry.)

In [ ]:
if SANDBOX:
    write(
        "b.json",
        [
            '{"device_id":"DEV-T-1","ts":"2026-11-01T08:10:00Z","heart_rate":90,"spo2":97,"co2_ppm":35}'
        ],
    )
    try:
        load()
    except Exception as error:  # UnknownFieldException: the schema was updated, run again
        print("first run stopped:", str(error).splitlines()[0][:200])
    load()
    display(spark.table(TABLE))

In [ ]:
# Clean up the sandbox (table, files, checkpoint and schema)
if SANDBOX:
    spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
    dbutils.fs.rm(TUTORIAL, recurse=True)

## 7. Try it yourself

**a)** Which simulated hour has the fewest readings in `bronze.vitals_raw`, and why might an hour have fewer? (Hint: dropped messages and battery outages in `generator/vitals.py`.)

In [ ]:
# Try it yourself
quiet_hour_sql = None

In [ ]:
# Answer (run after trying)
quiet_hour_sql = f"""
    SELECT date_trunc('hour', ts) AS hour, count(*) AS readings
    FROM {CATALOG}.bronze.vitals_raw GROUP BY 1 ORDER BY 2 LIMIT 3
"""
display(spark.sql(quiet_hour_sql))  # also: fewer patients on the ward at night

**b)** The outcomes files carry `recorded_ts = event_ts + 6 h`. Check that this holds for every row in `bronze.outcomes_raw`, and explain why the model can't use an outcome before its `recorded_ts`.

In [ ]:
# Try it yourself
delay_sql = None

In [ ]:
# Answer (run after trying)
delay_sql = f"""
    SELECT count(*) AS outcomes,
           count_if(recorded_ts = event_ts + INTERVAL 6 HOURS) AS six_hours_later
    FROM {CATALOG}.bronze.outcomes_raw
"""
display(spark.sql(delay_sql))

**c)** Without running it: if you re-upload a file with the **same name but different content**, does Auto Loader read it again? (It doesn't, unless `cloudFiles.allowOverwrites` is on. That's why reloading *different* data needs a full refresh of the pipeline.)

## 8. Pitfalls

**Pitfall: don't set `schemaLocation` or checkpoints in a pipeline.** Lakeflow manages both; you only set them when running Auto Loader yourself, as in the sandbox.

**Pitfall: reloading different data.** Auto Loader skips files it has seen, even if they changed. After `generator.load --replace` with a different seed, run the pipeline with a **full refresh**.

**Pitfall: inference without hints.** Types guessed from the first files can be wrong for later ones. Hint every field you know.

**Next:** notebook 12 turns Bronze into Silver: checks, quarantine, de-duplication, linking readings to patients, and history with AUTO CDC.